# Kernel Ablation: RBF vs Relation-Aware

**Goal:** Show contribution of relation-aware kernel design

**Comparison:**
- GP-KGE (RBF): Standard RBF kernel, no graph structure
- GP-KGE (Relation-Aware): Uses relation-specific graph Laplacians

**Quick Mode (CPU):** epochs=20, dim=100, MRR sample=1000 (~1hr on CPU)

In [ ]:
# Setup
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
import gc, json, warnings, time
import torch
import torch.nn.functional as F
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import eigsh
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models import GPKGE
from src.kernels.matern_graph import GraphLaplacian
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Quick mode config for CPU
CONFIG = {
    'embedding_dim': 100,  # 200 -> 100
    'epochs': 20,          # 50 -> 20
    'num_inducing': 300,   # 500 -> 300
    'mrr_sample': 1000,    # full -> 1000
    'eigendecomp_k': 50,   # 100 -> 50
}
print(f"Config: {CONFIG}")

train_data, _, test_data = load_fb15k237()
print(f"Data: {len(train_data):,} train, {len(test_data):,} test")

results = {}

In [ ]:
def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def evaluate(model, name):
    """Evaluation with sampling for speed"""
    print(f"\nEvaluating {name}...")
    model.eval()
    
    # MRR (sampled)
    sample_idx = np.random.choice(len(test_data), min(CONFIG['mrr_sample'], len(test_data)), replace=False)
    sample = test_data.triples[sample_idx]
    
    ranks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sample), 100), desc="MRR", leave=False):
            batch = sample[i:i+100]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model.score_tails(h, r)
            target = scores[torch.arange(len(t), device=device), t]
            ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())
    
    ranks = torch.tensor(ranks, dtype=torch.float)
    mrr = (1/ranks).mean().item()
    h1 = (ranks <= 1).float().mean().item()
    h10 = (ranks <= 10).float().mean().item()
    
    # ECE (sampled)
    ece_sample = min(2000, len(test_data))
    ece_idx = np.random.choice(len(test_data), ece_sample, replace=False)
    pos = test_data.triples[ece_idx]
    neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
    all_t = np.vstack([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])
    
    confs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(all_t), 1024), desc="ECE", leave=False):
            batch = all_t[i:i+1024]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model.score_triple(h, r, t)
            confs.append(torch.sigmoid(scores).cpu().numpy())
    conf = np.concatenate(confs)
    ece, _ = expected_calibration_error(conf, labels)
    brier = brier_score(conf, labels)
    
    # AUROC (sampled)
    ood_sample = min(2000, len(test_data))
    id_idx = np.random.choice(len(test_data), ood_sample, replace=False)
    id_triples = test_data.triples[id_idx]
    ood_triples = create_ood_dataset(train_data, test_data, "random", ood_sample)
    
    def get_unc(triples):
        uncs = []
        with torch.no_grad():
            for i in range(0, len(triples), 1024):
                batch = triples[i:i+1024]
                h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
                pred = model.predict_with_uncertainty(h, r, t)
                uncs.append(pred['total'].cpu().numpy())
        return np.concatenate(uncs)
    
    auroc = compute_auroc(get_unc(id_triples), get_unc(ood_triples))
    
    results[name] = {"mrr": mrr, "hits@1": h1, "hits@10": h10,
                     "ece": ece, "brier": brier, "auroc": auroc}
    print(f"{name}: MRR={mrr:.4f}, H@1={h1:.4f}, H@10={h10:.4f}, ECE={ece:.4f}, AUROC={auroc:.4f}")
    return results[name]

---
## GP-KGE (RBF Kernel)
---

In [ ]:
print("="*50 + "\nGP-KGE (RBF Kernel)\n" + "="*50)
set_seed(42)
clear_mem()

start = time.time()
model_rbf = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=CONFIG['embedding_dim'],
    kernel_type="rbf",
    num_inducing=CONFIG['num_inducing']
).to(device)

opt = torch.optim.Adam(model_rbf.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(CONFIG['epochs']), desc="GP-KGE (RBF)")):
    model_rbf.train()
    loss_sum, n = 0, 0
    for st in range(0, len(train_data), 1024):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()
        ps = model_rbf.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        ns = model_rbf.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_rbf.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

print(f"Training time: {time.time()-start:.1f}s")
evaluate(model_rbf, "GP-KGE (RBF)")
del model_rbf
clear_mem()

---
## GP-KGE (Relation-Aware Kernel)
---

In [ ]:
print("="*50 + "\nGP-KGE (Relation-Aware Kernel)\n" + "="*50)
set_seed(42)
clear_mem()

start = time.time()
model_ra = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=CONFIG['embedding_dim'],
    kernel_type="relation_aware",
    num_inducing=CONFIG['num_inducing']
).to(device)

# Eigendecomposition for relation-aware kernel
print("Computing eigendecomposition...")
kernel = model_ra.kernel
kernel.num_entities = train_data.num_entities
kernel.relation_laplacians = {}

success, failed = 0, 0
for rel_id, adj in tqdm(train_data.relation_adjacencies.items(), desc="Eigendecomp"):
    if adj.nnz < 10:
        continue
    try:
        degrees = np.array(adj.sum(axis=1)).flatten()
        D_inv_sqrt = sparse.diags(1.0 / np.sqrt(np.maximum(degrees, 1e-10)))
        L = sparse.diags(degrees) - adj
        L_norm = D_inv_sqrt @ L @ D_inv_sqrt
        L_norm = (L_norm + L_norm.T) / 2
        k = min(CONFIG['eigendecomp_k'], L_norm.shape[0] - 2)
        if k < 2:
            continue
        eigvals, eigvecs = eigsh(L_norm, k=k, which='SM', maxiter=500, tol=1e-3)
        kernel.relation_laplacians[rel_id] = GraphLaplacian(adj.shape[0])
        kernel.relation_laplacians[rel_id].eigenvalues = torch.tensor(eigvals, dtype=torch.float32)
        kernel.relation_laplacians[rel_id].eigenvectors = torch.tensor(eigvecs, dtype=torch.float32)
        success += 1
    except:
        failed += 1
print(f"Eigendecomp: {success} success, {failed} failed")

In [ ]:
# Training
opt = torch.optim.Adam(model_ra.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(CONFIG['epochs']), desc="GP-KGE (RA)")):
    model_ra.train()
    loss_sum, n = 0, 0
    for st in range(0, len(train_data), 1024):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()
        ps = model_ra.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        ns = model_ra.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ra.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

print(f"Training time: {time.time()-start:.1f}s")
evaluate(model_ra, "GP-KGE (Relation-Aware)")
del model_ra
clear_mem()

---
## Results
---

In [ ]:
print("\n" + "="*70)
print("KERNEL ABLATION RESULTS")
print("="*70)
print(f"{'Kernel':<25} {'MRR':>8} {'H@1':>8} {'H@10':>8} {'ECE':>8} {'AUROC':>8}")
print("-"*70)

for name, r in results.items():
    print(f"{name:<25} {r['mrr']:>8.4f} {r['hits@1']:>8.4f} {r['hits@10']:>8.4f} {r['ece']:>8.4f} {r['auroc']:>8.4f}")

if len(results) == 2:
    rbf = results.get("GP-KGE (RBF)", {})
    ra = results.get("GP-KGE (Relation-Aware)", {})
    if rbf and ra:
        print("\n" + "="*70)
        print("IMPROVEMENT (Relation-Aware vs RBF)")
        print("="*70)
        print(f"  MRR:   {rbf['mrr']:.4f} -> {ra['mrr']:.4f} ({(ra['mrr']-rbf['mrr'])/rbf['mrr']*100:+.1f}%)")
        print(f"  H@10:  {rbf['hits@10']:.4f} -> {ra['hits@10']:.4f} ({(ra['hits@10']-rbf['hits@10'])/rbf['hits@10']*100:+.1f}%)")
        print(f"  ECE:   {rbf['ece']:.4f} -> {ra['ece']:.4f} ({(rbf['ece']-ra['ece'])/rbf['ece']*100:+.1f}% better)")

In [ ]:
# Save
with open('kernel_ablation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved to kernel_ablation_results.json")

try:
    from google.colab import files
    files.download('kernel_ablation_results.json')
except:
    pass